# Iris : score affine et MLP

Ce notebook accompagne l'exercice 6.1. Nous commençons avec les deux mesures des pétales afin de voir les régions de décision, puis nous utilisons les quatre mesures.

Les valeurs produites par softmax seront appelées **poids normalisés**. Leur somme vaut un, mais cette propriété algébrique ne suffit pas à définir un modèle probabiliste sous-jacent.

## Parcours

1. [Données et partition](#donnees-iris)
2. [Score affine et régions polyédriques](#affine-iris)
3. [Invariance des scores](#invariance-iris)
4. [MLP et frontières non affines](#mlp-iris)
5. [Fréquences et matrices de confusion](#confusion-iris)
6. [Retour aux quatre mesures](#quatre-mesures-iris)

In [1]:
import jax
import jax.numpy as jnp
import flax
from flax import nnx
import optax
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

jax.config.update("jax_enable_x64", True)
print(
    f"JAX {jax.__version__}, Flax {flax.__version__}, "
    f"Optax {optax.__version__}, scikit-learn {sklearn.__version__}"
)

JAX 0.11.1, Flax 0.12.9, Optax 0.2.8, scikit-learn 1.9.0


<a id="donnees-iris"></a>
## 1. Données et partition

Chargez le jeu `Iris`. Conservez d'abord la longueur et la largeur du pétale. Représentez les trois espèces, puis construisez une partition déterministe 60 % / 20 % / 20 % en conservant leurs fréquences.

Centrez et réduisez les caractéristiques en utilisant uniquement les données d'apprentissage. Cette précaution évite que les données de validation ou de test interviennent dans la construction de la machine.

In [ ]:
# À compléter.

<a id="affine-iris"></a>
## 2. Score affine et régions polyédriques

Entraînez un score affine $x\mapsto Ax+b\in\mathbb R^3$ avec l'entropie croisée. Choisissez $\lambda\in\{0,10^{-4},10^{-3},10^{-2}\}$ à partir de l'erreur de validation.

Représentez les régions de décision. Pour une classe $a$, vérifiez sur la grille les inégalités
$$(A_a-A_c)x\geq b_c-b_a,\qquad c\ne a.$$

In [3]:
class ScoreAffine(nnx.Module):
    def __init__(self, n_entrees, n_classes, *, rngs):
        self.sortie = nnx.Linear(n_entrees, n_classes, rngs=rngs)

    def __call__(self, x):
        return self.sortie(x)


class MLP(nnx.Module):
    def __init__(self, n_entrees, largeur, n_classes, *, rngs):
        self.W1 = nnx.Linear(n_entrees, largeur, rngs=rngs)
        self.W2 = nnx.Linear(largeur, n_classes, rngs=rngs)

    def __call__(self, x):
        return self.W2(nnx.relu(self.W1(x)))


def norme_parametres(machine):
    etat = nnx.state(machine, nnx.Param)
    return sum(jnp.sum(feuille**2) for feuille in jax.tree.leaves(etat))


def perte(machine, x, z, lamb=0.0):
    scores = machine(x)
    ce = optax.softmax_cross_entropy_with_integer_labels(scores, z).mean()
    return ce + 0.5 * lamb * norme_parametres(machine)


@nnx.jit
def pas(machine, optimizer, x, z, lamb):
    valeur, gradient = nnx.value_and_grad(perte)(machine, x, z, lamb)
    optimizer.update(machine, gradient)
    return valeur


def entrainer(machine, x, z, lamb=0.0, alpha=0.03, iterations=700):
    optimizer = nnx.Optimizer(machine, optax.adam(alpha), wrt=nnx.Param)
    for _ in range(iterations):
        valeur = pas(machine, optimizer, x, z, lamb)
    return float(valeur)


def frequence_erreur(machine, x, z):
    predictions = np.asarray(jnp.argmax(machine(jnp.asarray(x)), axis=1))
    return float(np.mean(predictions != np.asarray(z)))


def nombre_parametres(machine):
    return sum(feuille.size for feuille in jax.tree.leaves(nnx.state(machine, nnx.Param)))

In [ ]:
# À compléter.

<a id="invariance-iris"></a>
## 3. Invariance des scores

Pour trois fleurs, affichez les scores, les poids normalisés et la classe prédite. Ajoutez ensuite la même constante aux trois scores et vérifiez l'invariance du softmax et de la décision.

In [ ]:
# À compléter.

<a id="mlp-iris"></a>
## 4. MLP et frontières non affines

Entraînez des MLP à une couche cachée de largeur 8 ou 16. Choisissez la largeur et l'initialisation à partir des données de validation, puis comparez la frontière retenue avec celle du score affine.

In [ ]:
# À compléter.

<a id="confusion-iris"></a>
## 5. Fréquences et matrices de confusion

Comparez les fréquences d'erreur sur les trois ensembles. Construisez les matrices de confusion brutes et normalisées sur les données de test.

In [ ]:
# À compléter.

<a id="quatre-mesures-iris"></a>
## 6. Retour aux quatre mesures

Reprenez l'expérience avec les quatre mesures. La représentation des régions dans le plan disparaît, mais les fréquences et les matrices de confusion restent définies.

In [ ]:
# À compléter.

## Bilan

Le score affine produit des régions polyédriques convexes. Le MLP peut courber ou décomposer ces régions, mais cette liberté supplémentaire n'est utile que si les données l'exigent. La matrice de confusion montre ici que les erreurs se concentrent entre *versicolor* et *virginica*.